In [ ]:
!pip install --upgrade tensorflow
!pip install mtcnn
!pip install dlib imutils

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt
import dlib

In [ ]:
ruta_modelo = '/content/drive/Shareddrives/ClasifEye/modelo_caras_resnet50_07_18.h5'
model = load_model(ruta_modelo, compile=False)
print("✅ Modelo cargado")

In [ ]:
ruta_imagen = '/content/drive/Shareddrives/ClasifEye/'
nombre_imagen = '23_0_3_20170116161243731.jpg'
ruta_imagen = ruta_imagen + nombre_imagen

#Preprocesar imagen

In [ ]:
from mtcnn import MTCNN
import cv2
import matplotlib.pyplot as plt

detector_mtcnn = MTCNN()

def recortar_cara(imagen):
    # Convertir la imagen a RGB para MTCNN
    imagen_corregida = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)

    # Detección de rostros
    resultados = detector_mtcnn.detect_faces(imagen_corregida)

    if len(resultados) == 0:
        print("No se detectó ningún rostro en la imagen.")
        plt.imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
        plt.show()
        return imagen, None, None  # Devolver imagen original y None para confidence y keypoints

    # Obtener el primer rostro detectado
    result = resultados[0]
    x, y, w, h = result['box']
    confidence = result['confidence']
    keypoints = result['keypoints']

    # Recortar la imagen
    imagen_recortada = imagen[y:y+h, x:x+w]

    return imagen_recortada, confidence, keypoints

In [ ]:
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -d shape_predictor_68_face_landmarks.dat.bz2

In [ ]:
SHAPE_PREDICTOR_PATH = "shape_predictor_68_face_landmarks.dat"
OUTPUT_SIZE = (224, 224)

# Inicializar el detector de rostros y el predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(SHAPE_PREDICTOR_PATH)

def align_and_resize_face(image: np.ndarray) -> np.ndarray:
    if image is None or not isinstance(image, np.ndarray):
        print("Imagen inválida. Devolviendo imagen negra redimensionada.")
        return cv2.resize(np.zeros((100, 100, 3), dtype=np.uint8), OUTPUT_SIZE)

    original_image = image.copy()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = detector(gray, 1)

    if len(faces) == 0:
        print("No se detectó ningún rostro. Redimensionando imagen original.")
        return cv2.resize(original_image, OUTPUT_SIZE)

    face = faces[0]
    landmarks = predictor(gray, face)
    landmarks_np = np.array([[p.x, p.y] for p in landmarks.parts()], dtype=np.float32)

    left_eye = np.mean(landmarks_np[36:42], axis=0)
    right_eye = np.mean(landmarks_np[42:48], axis=0)

    # Calcular ángulo
    dx = right_eye[0] - left_eye[0]
    dy = right_eye[1] - left_eye[1]
    angle = np.degrees(np.arctan2(dy, dx))

    # Calcular centro entre los ojos
    eyes_center = ((left_eye[0] + right_eye[0]) / 2,
                   (left_eye[1] + right_eye[1]) / 2)

    # Distancia entre ojos en la imagen original
    dist = np.sqrt(dx**2 + dy**2)

    # Distancia deseada entre ojos
    desired_eye_dist = OUTPUT_SIZE[0] * (0.68 - 0.32)
    scale = desired_eye_dist / dist

    # Obtener matriz de rotación + escala
    M = cv2.getRotationMatrix2D(eyes_center, angle, scale)

    # Mover los ojos a la posición deseada
    tx = OUTPUT_SIZE[0] * 0.5 - eyes_center[0]
    ty = OUTPUT_SIZE[1] * 0.4 - eyes_center[1]
    M[0, 2] += tx
    M[1, 2] += ty

    aligned = cv2.warpAffine(image, M, OUTPUT_SIZE, flags=cv2.INTER_LINEAR)

    return aligned

In [ ]:
import matplotlib.pyplot as plt

def procesar_imagen(imagen):
    #plt.imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
    #plt.show()

    # Aplicar el pipeline
    imagen_procesada, confidence, keypoints = recortar_cara(imagen)
    imagen_procesada = align_and_resize_face(imagen_procesada)
    imagen_procesada, confidence, keypoints = recortar_cara(imagen_procesada)

    return imagen_procesada, confidence, keypoints

#Predicción

In [ ]:
img = image.load_img(ruta_imagen, target_size=(224, 224))

img_array = image.img_to_array(img)
img_preprocessed = preprocess_input(img_array)
img_input = np.expand_dims(img_preprocessed, axis=0)

In [ ]:
img = image.load_img(ruta_imagen, target_size=(224, 224))
img_np = np.array(img)
img_final, confidence, keypoints = procesar_imagen(img_np)

# Ajustar dimensiones de la img y los keypoints de entrada
img_input = np.expand_dims(img_preprocessed, axis=0)

valores = list(keypoints.values())  # Extrae solo los valores en orden
valores = np.array(valores, dtype=np.float32) / 224.0
keypoints_np = np.array(valores, dtype='float32')  # (10,)
keypoints_input = np.expand_dims(keypoints_np, axis=0)  # (1, 10)

keypoints_reshaped = keypoints_input.reshape(1, 10)  # convierte (1, 5, 2) → (1, 10)

In [ ]:
print("Forma de la imagen:", img_input.shape)  # Esperado: (1, 224, 224, 3)

In [ ]:
print("Forma de los keypoints:", keypoints_reshaped.shape)  # Esperado: (1, 10)

In [ ]:
pred = model.predict([img_input, keypoints_reshaped])

In [ ]:
pred

In [ ]:
# Mostrar imagen original (no preprocesada)
plt.imshow(img)
plt.axis('off')  # Oculta ejes
plt.title("Imagen de entrada")
plt.show()

In [ ]:
# Escalar edad

# Interpretar sexo
sexo_labels = ['Hombre', 'Mujer']
sexo_pred = sexo_labels[np.argmax(pred[1][0])]

# Interpretar etnia

# Mostrar resultados
print(f"Género: {sexo_pred}")

In [ ]:
y_cols_edad = ['edad']
y_cols_sexo = ['sexo_Hombre', 'sexo_Mujer']
y_cols_raza = ['raza_Asiatico', 'raza_Blanco', 'raza_Indio', 'raza_Negro', 'raza_Otros']

edad = pred[0][0][0] * 115

# Interpretar sexo
sexo_idx = np.argmax(pred[1][0])
sexo = y_cols_sexo[sexo_idx].replace("sexo_", "")

# Interpretar raza
raza_idx = np.argmax(pred[2][0])
raza = y_cols_raza[raza_idx].replace("raza_", "")

# Mostrar resultados
print(f"🧓 Edad estimada: {edad:.2f} años")
print(f"⚧️ Sexo estimado: {sexo}")
print(f"🌍 Raza estimada: {raza}")

### **Prueba de predicciones con 10 imágenes**

In [ ]:
import os

image_dir = '/content/drive/Shareddrives/ClasifEye/Dataset Preprocesado'

try:
    all_files = os.listdir(image_dir)
    image_files = [f for f in all_files if f.lower().endswith('.jpg')]
    print(f"Found {len(image_files)} .jpg files.")
except FileNotFoundError:
    print(f"Error: Directory not found at {image_dir}")
    image_files = []

In [ ]:
import random

if len(image_files) >= 10:
    selected_images = random.sample(image_files, 10)
    print("Selected 10 random images:")
    for img_name in selected_images:
        print(f"- {img_name}")
else:
    print(f"Not enough images found. Found {len(image_files)}, but need 10.")
    selected_images = image_files

In [ ]:
for img_name in selected_images:
    full_image_path = os.path.join(image_dir, img_name)
    print(f"\nProcesando imagen: {img_name}")

    try:
        # Cargar y preprocesar la imagen
        img = cv2.imread(full_image_path)
        if img is None:
            print(f"Error al cargar la imagen: {img_name}")
            continue

        # Usar .copy() para evitar modificar la imagen original
        img_processed, confidence, keypoints = procesar_imagen(img.copy())

        if img_processed is None:
            print(f"Saltando {img_name} debido a un error de preprocesamiento.")
            continue

        # Preprocesar para la entrada del modelo
        img_final_rgb = cv2.cvtColor(img_processed, cv2.COLOR_BGR2RGB)
        img_array = image.img_to_array(img_final_rgb)
        img_preprocessed = preprocess_input(img_array)
        img_input = np.expand_dims(img_preprocessed, axis=0)

        # Preparar la entrada de keypoints
        if keypoints and len(keypoints) == 5:
            valores = list(keypoints.values())  # Extrae solo los valores en orden
            valores = np.array(valores, dtype=np.float32) / 224.0 # Escalar keypoints
            keypoints_np = np.array(valores, dtype='float32')  # (10,)
            keypoints_input = np.expand_dims(keypoints_np, axis=0)  # (1, 10)
            keypoints_reshaped = keypoints_input.reshape(1, 10) # Asegurar forma (1, 10)
        else:
             print(f"Advertencia: Keypoints no detectados o en formato inesperado para {img_name}. Usando keypoints falsos.")
             # Crear keypoints falsos si no se detectan o están en formato incorrecto
             keypoints_reshaped = np.zeros((1, 10), dtype='float32')


        # Realizar la predicción
        pred = model.predict([img_input, keypoints_reshaped])

        # Interpretar y mostrar resultados
        edad = pred[0][0][0] * 115

        sexo_idx = np.argmax(pred[1][0])
        sexo = y_cols_sexo[sexo_idx].replace("sexo_", "")

        raza_idx = np.argmax(pred[2][0])
        raza = y_cols_raza[raza_idx].replace("raza_", "")

        print(f"🧓 Edad estimada: {edad:.2f} años")
        print(f"⚧️ Sexo estimado: {sexo}")
        print(f"🌍 Raza estimada: {raza}")

        # Opcionalmente, mostrar la imagen precesada
        plt.imshow(img_final_rgb)
        plt.axis('off')
        plt.title(f"Procesada: {img_name}")
        plt.show()

    except Exception as e:
        print(f"Ocurrió un error al procesar {img_name}: {e}")

print("\nTerminado el procesamiento de las imágenes seleccionadas.")